This notebook prcesses raw skeletons by breaking branches, and remerging the fragments. After all strips from a given directory are processed, they are then combined using translation and offset information.

In [1]:
import zarr
import numpy as np
import matplotlib.pyplot as plt
import navis
import os
import requests
import glob
from cloudvolume import CloudVolume, Skeleton
import numpy as np
from pathlib import Path
from io import BytesIO

import concurrent.futures
from joblib import dump,load, Parallel, delayed, parallel_config
from natsort import natsorted
import uuid
import random
import pandas as pd
from ac_segmentation.reconnect_stack_navis import reconnect, read_navis_neurons_tar, write_navis_skels_tar, remove_cutout_nodes, translate_nodes, create_rectangle_volume, remove_overlap_nodes

In [ ]:
#Set parameters
n_jobs = 10 #number of parallel jobs
im_shape = [22848,288,288] #strip shape
sc = load("/ACdata/Users/connorl/Models/scaler.joblib") #scalar file
cl = load("/ACdata/Users/connorl/Models/LR_1.joblib") #model file
in_dir = "/allen/programs/celltypes/workgroups/em-connectomics/russelt/ac_processing_demo/data/skeleton_outputs/H17_x55_S33_230413_highres.zarr/"
out_dir = "/ACdata/Users/connorl/Skeletons/H17_x55_S33_230413_highres_MIP1_test/"

#Load translation data
file_trans = pd.read_json('/ACdata/Users/connorl/Skeletons/H17_x55_S33_230413_highres_MIP1_test/S33_file_trans.json', orient='records', lines=True)
file_trans.columns = ['File','Translation']

#Match original files to output files
out_fn = {}
for ind,row in file_trans.iterrows():
    rem = os.path.dirname(row['File'])+"/"
    f = row['File'].replace(rem, out_dir)
    out_fn[row['File']] = f

In [ ]:
from scipy.spatial import distance
def find_overlap_volumes(files, translations, im_shape):
    vertices,vols = [],[]
    translate_df = {files[i]: translations[i] for i in range(len(files))}
    for file,trans in translate_df.items():
        #translate dimensions
        sx,sy,sz = np.array(im_shape)
        tx,ty,tz = trans
        post = np.array([0,sx,0,sy,0,sz]) + np.array([tx,tx,ty,ty,tz,tz])
        #create volume
        vol = create_rectangle_volume(post, file)
        vols.append(vol), vertices.append(vol.vertices.tolist())
    
    matches = []
    for file,vert in zip(files,vertices):
        #test keypoints against all volumes
        result = navis.in_volume(vert, vols)
        #save matches
        for ind,(key,value) in enumerate(result.items()):
            if any(value) == True:
                if file == key:
                    continue
                #gauge overlap
                overlap = abs(vert[0]-vols[ind].vertices[0])
                overlap[overlap!=0] = 1
                #sort and combine with overlap
                match = [file,key]
                match.sort()
                match += [tuple(overlap.astype('int'))]
                if match not in matches:
                    matches.append(match)
    volumes = pd.DataFrame(zip(files,vols), columns=['File','Vol'])
    matches = pd.DataFrame(matches, columns=['File1','File2','xyz_overlap'])

    #arrange match order according to priximity to origin
    for ind,row in matches.iterrows():
        p1, p2 = translate_df[row['File1']], translate_df[row['File2']]
        dis = [distance.euclidean((0,0,0), tuple(p1)), distance.euclidean((0,0,0), tuple(p2))]
        dis = dis.index(min(dis))
        if dis == 1:
            rep = {'File1':row['File2'],'File2':row['File1'], 'xyz_overlap':row['xyz_overlap']}
            matches.loc[ind,rep.keys()] = list(rep.values())
    
    return matches,volumes

files, translations = list(file_trans['File']), list(file_trans['Translation'])
matches,volumes = find_overlap_volumes(files=files, translations=translations , im_shape=im_shape)

In [ ]:
###Order the matches so there are no duplicate files in a job batch
def order_matches(matches, n_jobs):
    store = matches.copy()
    ordered = []
    attempts = 0
    while ((attempts<n_jobs*1000) and (len(store)>=1)):
        try:
            sample = store.sample(n_jobs)
        except:
            sample = store.sample(len(store))
        fns = list(sample['File1'])+list(sample['File2'])
        if (len(fns) != len(set(fns))) == False:
            for ind,row in sample.iterrows():
                ordered.append(list(row))
            store = store.drop(list(sample.index))
            store.reset_index()
        attempts+=1
    ordered += store.values.tolist()
    matches = pd.DataFrame(ordered, columns=['File1','File2','xyz_overlap'])
    return matches

matches =  order_matches(matches, 5)

In [ ]:
###Find boundaries of overlap
def find_overlap_bounds(matches,volumes):
    file_overlap = {}
    #set empty values
    for ind,row in matches.iterrows():
        file_overlap[row['File1']] = []
        file_overlap[row['File2']] = []
        
    for ind,row in matches.iterrows():
        over_dim = [(0,0),(0,0),(0,0)]
        v1 = volumes[volumes['File']==row['File1']]['Vol'].item().bbox
        v2 = volumes[volumes['File']==row['File2']]['Vol'].item().bbox
        v1d = np.around(v1.T,1)
        for ind,dim in enumerate(row['xyz_overlap']):
            if dim != 0:
                d1 = np.around(v1.T[ind],1)
                d2 = np.around(v2.T[ind],1)
                r1 = np.around(np.arange(d1[0],d1[1], .1),1)
                r2 = np.around(np.arange(d2[0],d2[1], .1),1)
                inter = list(set(r1).intersection(r2))
                inter.sort()
                inter = inter[0],inter[-1]
                over_dim[ind]=inter
    
        #skip if corner interesection
        res = sum(1 for i in over_dim if i == (0,0))
        if res == 1:
            continue
        #if no overlap, replace with standard boundary
        for ind,dim in enumerate(over_dim):
            if dim == (0,0):
                over_dim[ind] = tuple(v1d[ind])
        file_overlap[row['File1']] = file_overlap[row['File1']] + [[i for sub in over_dim for i in sub]]

    return file_overlap

bounds = find_overlap_bounds(matches,volumes)

In [ ]:
###Reconnect, translate, and filter individual strips
def postprocess_strip(out_dir, file, cl, sc, bound_boxs=None, trans=[0,0,0], min_nodes=10, query_dis=20, min_collin=.7, resample=4, smooth=2):
    #translate strip
    print(file)
    skels = read_navis_neurons_tar(file)
    fname = file.split("/")[-1]
    if trans != [0,0,0]:
        skels = translate_nodes(skels, trans=trans)
    
    #remove overlapping nodes
    if bound_boxs != None:
        for bb in bound_boxs:
            skels = remove_cutout_nodes(skels, bound_box=bb)
    
    #reconnect skeletons
    skels, merges = reconnect(skels=skels, cl=cl, sc=sc, min_nodes=min_nodes, query_dis=query_dis, min_collin=min_collin, resample=resample, smooth=smooth, split=True)
    skels = navis.NeuronList([skels,merges])
    skels, merges = reconnect(skels=skels, cl=cl, sc=sc, min_nodes=min_nodes, query_dis=10, min_collin=min_collin, resample=None, smooth=None, split=False)
    skels = navis.NeuronList([skels,merges])
    write_navis_skels_tar(out_dir+fname, skels)

with parallel_config(backend="loky", inner_max_num_threads=1):
    %time res = Parallel(n_jobs=n_jobs)(delayed(postprocess_strip)(out_dir=out_dir, file=row['File'], cl=cl, sc=sc, bound_boxs=bounds[row['File']], trans=row['Translation']) for ind,row in file_trans.iterrows())

In [ ]:
###Reconnect adjacent strips
def reconnect_strips(strip1, strip2, overlap, cl=None, sc=None, edge_prop=.8, min_nodes=20, query_dis=20, min_collin=0.6, prob_thresh=0.5, dis_end=0):
    if isinstance(strip1, navis.core.neuronlist.NeuronList):
        s1,s2 = strip1,strip2
        strip1,strip2 = 'strip1','strip2'
        pass
    else:
      if strip1.endswith('.gz'):
        s1 = read_navis_neurons_tar(strip1)
        s2 = read_navis_neurons_tar(strip2)
      elif strip1.endswith('.swc'):
        s1 = navis.read_swc(strip1)
        s2 = navis.read_swc(strip2)
          
    #combine neuronlist pairs and set strip names
    s1.set_neuron_attributes([strip1]*int(len(s1)), 'strip')
    s2.set_neuron_attributes([strip2]*int(len(s2)), 'strip')
    skels = navis.NeuronList([s1,s2])

    #find bounding box
    edge_dis = [edge_prop if i != 0  else i for i in overlap]
    x,y,z = skels.bbox
    x_dis,y_dis,z_dis = ((np.diff(x)*edge_dis[0])/2)[0], ((np.diff(y)*edge_dis[1])/2)[0], ((np.diff(z)*edge_dis[2])/2)[0]
    bound_box = np.concatenate((x + np.array([x_dis,-x_dis]), y + np.array([y_dis,-y_dis]), z + np.array([z_dis,-z_dis]))).tolist()

    #run reconnection
    non_merged, merged = reconnect(skels=skels, cl=cl, sc=sc, min_nodes=min_nodes, query_dis=query_dis, min_collin=min_collin, 
                                   prob_thresh=prob_thresh, resample=None, smooth=None, split=False, bound_box=bound_box, dis_end=dis_end)
    merged.set_neuron_attributes(['merged']*int(len(merged)), 'strip')

    if not isinstance(s1, navis.core.neuronlist.NeuronList):
        #label nodes by strip
        for sk in non_merged:
            sk.nodes['label']=sk.strip
        for sk in merged:
            sk.nodes['label']='merge'
        #replace strip files
        for ind,strip in enumerate(list(set(non_merged.strip))):
            strip_sk = [i for i, value in enumerate(non_merged.strip) if value==strip]
            subset = non_merged[strip_sk]
            subset = navis.NeuronList([subset,merged])

            os.remove(strip)
            write_navis_skels_tar(strip, subset)
    
    return non_merged,merged

#with parallel_config(backend="loky", inner_max_num_threads=1):
    #%time res = Parallel(n_jobs=n_jobs)(delayed(reconnect_strips)(strip1=out_fn[row['File1']] , strip2=out_fn[row['File2']], overlap=row['xyz_overlap'], cl=cl, sc=sc, outfile=True) for ind,row in matches.iterrows())

In [ ]:
###Pull out merged skeletons
def get_merged_skeletons(file):
    if isinstance(file, navis.core.neuronlist.NeuronList):
        skels = file
        pass
    else:
      if file.endswith('.gz'):
        skels = read_navis_neurons_tar(file)
      elif file.endswith('.swc'):
        skels = navis.read_swc(file)
          
   non_merged, merged = navis.NeuronList(None), navis.NeuronList(None)
    for sk in skels:
        if sk.nodes.iloc[0]['label']=="'merged'":
            merged.append(sk)
        else:
            non_merged.append(sk)

    os.remove(file)
    write_navis_skels_tar(file, non_merged)
    return merged

with parallel_config(backend="loky", inner_max_num_threads=1):
    %time merged_skels = Parallel(n_jobs=n_jobs)(delayed(get_merged_skeletons)(file=out_fn[row['File']]) for ind,row in file_trans.iterrows())

In [ ]:
###Remove overlapping nodes and run final reconnection
merged_skels = navis.NeuronList(merged_skels)
merged_skels = navis.remove_overlap_nodes(merged_skels)
non_merged,merged =  reconnect(skels=merged_skels, cl=cl, sc=sc, min_nodes=10, query_dis=10, min_collin=0.7, 
                                   prob_thresh=.1, resample=None, smooth=None, split=False)
write_navis_skels_tar(out_dir+'merged.swcs.tar.gz', navis.NeuronList([non_merged,merged]))

In [ ]:
###Run final reconnection
non_merged,merged =  reconnect(skels=merged_skels, cl=cl, sc=sc, min_nodes=10, query_dis=10, min_collin=0.7, 
                                   prob_thresh=.1, resample=None, smooth=None, split=False)
write_navis_skels_tar(out_dir+'merged.swcs.tar.gz', navis.NeuronList([non_merged,merged]))